# Credit Risk Model Fix - Complete Submission

This notebook identifies and fixes 14 critical machine learning issues including data leakage, improper validation, and incorrect evaluation.

## Bug Identification and Fixes (14 Total)

1. Incorrect import of SimpleImputer → fixed import
2. Leakage: loan_status_final → removed
3. Leakage: repayment_flag → removed
4. Leakage: last_payment_status → removed
5. Engineered leakage: risk_indicator → removed
6. Engineered leakage: payment_behavior_score → removed
7. Feature selection on full dataset → avoided
8. Preprocessing fit on full data → fixed using pipeline
9. Noise features included → removed
10. Hyperparameter tuning on test set → replaced with cross-validation
11. Threshold tuning on test set → avoided
12. No cross-validation → added Stratified K-Fold
13. Single model used → baseline comparison added
14. Feature importance mismatch → handled via pipeline

All issues are fixed in the implementation below.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, classification_report

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv("credit_risk_dataset.csv")

print("Shape:", df.shape)
print(df['target_flag'].value_counts(normalize=True))


In [ ]:
leakage_cols = [
    'loan_status_final',
    'repayment_flag',
    'last_payment_status',
    'risk_indicator',
    'payment_behavior_score'
]

noise_cols = ['random_score_1', 'random_score_2', 'duplicate_feature']

df = df.drop(columns=leakage_cols + noise_cols, errors='ignore')


In [ ]:
X = df.drop("target_flag", axis=1)
y = df["target_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    scores = cross_validate(pipeline, X_train, y_train, cv=cv,
                            scoring=['roc_auc', 'f1'])

    print(name)
    print("ROC-AUC:", scores['test_roc_auc'].mean())
    print("F1:", scores['test_f1'].mean())
    print("-"*40)


In [ ]:
final_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

final_pipeline.fit(X_train, y_train)


In [ ]:
y_pred_proba = final_pipeline.predict_proba(X_test)[:,1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("F1:", f1_score(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
